# 20. Development-Selected Dual-Head Epoch 2 — Repaired-Test Descriptive Comparison

This notebook performs one frozen post-development descriptive evaluation of the
development-selected residual dual-head **epoch-2** checkpoint.

It reuses all established repaired-test predictions and generates only:

1. dual-head epoch 2;
2. dual-head epoch 2 + Hard-v2.

The notebook compares eight arms: fusion-only, shared selector, dual epoch 1,
dual epoch 2, and the corresponding Hard-v2 systems.


## 1. Protocol

Frozen before this run:

- dual checkpoint: epoch 2 selected by notebook 19 using development-only
  trigger-budget matching;
- selector threshold: `0.70`;
- compatibility margin: `8.0`;
- Hard-v2 minimum depth: `2`;
- Hard-v2 escape margin: `10.0`.

The repaired test is already part of the broader development history.
Therefore, this run is **post-development descriptive**, not pristine
confirmatory evidence. Do not retune after viewing the output.


## 2. Environment

In [ ]:
# 1. Bootstrap (Xet disabled BEFORE any HF import — lesson from 12b).
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip -q install "transformers>=4.44,<5" "huggingface_hub>=0.25,<1" nltk rouge-score accelerate sentencepiece sacremoses sacrebleu

import sys, json, pickle, random, time, shutil, subprocess, re, unicodedata, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
drive.mount('/content/drive')
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required'
import pandas as pd
from IPython.display import display
import zipfile


In [ ]:
# 1.1 PyTorch Geometric.
try:
    import torch_geometric
    print('ok:', torch_geometric.__version__)
except Exception:
    tv = torch.__version__.split('+')[0]; cv = torch.version.cuda
    url = f'https://data.pyg.org/whl/torch-{tv}+cu{cv.replace(".", "")}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', url])
    import torch_geometric

## 3. Paths and frozen configuration

In [ ]:
# Project paths. Change only PROJECT_DIR if your Drive layout differs.
PROJECT_DIR = '/content/drive/MyDrive/kg_llm_project'
PROCESSED_DIR = f'{PROJECT_DIR}/baseline-bart-webnlg/processed'
CKPT_FUSION = f'{PROJECT_DIR}/fusion_only_outputs/checkpoints/fusion_only/model_best.pt'
CURRENT_SELECTOR_CKPT = f'{PROJECT_DIR}/selector_eval_v5_earlystop/selector_best_v5.pt'

# Frozen archives.
OPERATING_POINT_ARCHIVE = f'{PROJECT_DIR}/dual_head_operating_point_dev_v1.zip'
PREVIOUS_DUAL_TEST_ARCHIVE = f'{PROJECT_DIR}/dual_head_repaired_test_eval_v1.zip'
SELECTOR_TEST_ARCHIVE = f'{PROJECT_DIR}/selector_test_descriptive_v2.zip'

OPERATING_POINT_EXTRACT_DIR = f'{PROJECT_DIR}/dual_head_operating_point_dev_v1_frozen'
PREVIOUS_DUAL_TEST_DIR = f'{PROJECT_DIR}/dual_head_repaired_test_eval_v1_frozen'
SELECTOR_TEST_DIR = f'{PROJECT_DIR}/selector_test_descriptive_v2_frozen'
OUTPUT_DIR = f'{PROJECT_DIR}/dual_head_epoch2_repaired_test_eval_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda'
MAX_GEN_LEN, MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 256, 128
MODEL_DIM = 768
CONSTRAINT_BOTTLENECK = 256
CONSTRAINT_NUM_BASES = 8
CONSTRAINT_DROPOUT = 0.10

# Frozen development-selected operating point.
SELECTOR_TAU = 0.70
COMPATIBILITY_MARGIN = 8.0
LOCK_MIN_DEPTH = 2
ESCAPE_MARGIN = 10.0

EXPECTED_EPOCH2_SHA = '3916ece3a811f1f5ecf1ba1d628467d7bda20a5e5065124a42a14b88c107fd10'
EXPECTED_EPOCH1_TEST_SHA_PREFIX = '7bc86efbad50'

BOOTSTRAP_SAMPLES = 2000
RUN_OPTIONAL_BLEU_BOOTSTRAP = False
BLEU_BOOTSTRAP_SAMPLES = 500
SAVE_EVERY = 25

required = [
    f'{PROJECT_DIR}/fixed_ablation_common.py',
    PROCESSED_DIR,
    CKPT_FUSION,
    CURRENT_SELECTOR_CKPT,
    OPERATING_POINT_ARCHIVE,
    PREVIOUS_DUAL_TEST_ARCHIVE,
    SELECTOR_TEST_ARCHIVE,
]
for path in required:
    assert os.path.exists(path), f'Missing required artifact: {path}'

print('Output directory:', OUTPUT_DIR)
print('Epoch-2 archive:', OPERATING_POINT_ARCHIVE)
print('Previous epoch-1 test archive:', PREVIOUS_DUAL_TEST_ARCHIVE)
print('Shared-selector test archive:', SELECTOR_TEST_ARCHIVE)


In [ ]:
shutil.copy(f'{PROJECT_DIR}/fixed_ablation_common.py', '/content/fixed_ablation_common.py')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import fixed_ablation_common as fac
from fixed_ablation_common import (
    load_artifacts, FusionOnlyGNNModel, load_variant_checkpoint_resume,
    add_final_logits_bias, pad_kg_nodes, find_entity_token_spans,
)

from urllib.parse import quote
BART_LOCAL = '/content/bart-base-local'; os.makedirs(BART_LOCAL, exist_ok=True)
FILES = {
    'config.json': 1000,
    'vocab.json': 800000,
    'merges.txt': 400000,
    'tokenizer.json': 1000000,
    'model.safetensors': 500000000,
}
for fn, mn in FILES.items():
    dest = os.path.join(BART_LOCAL, fn)
    if os.path.isfile(dest) and os.path.getsize(dest) >= mn:
        continue
    url = f'https://huggingface.co/facebook/bart-base/resolve/main/{quote(fn)}?download=true'
    rc = subprocess.run([
        'curl','-L','--fail','--retry','12','--retry-delay','5','--retry-all-errors',
        '--connect-timeout','30','--speed-time','90','--speed-limit','1024',
        '-C','-','-o',dest + '.part',url,
    ]).returncode
    if rc != 0:
        if os.path.exists(dest + '.part'): os.remove(dest + '.part')
        rc = subprocess.run(['curl','-L','--fail','-o',dest + '.part',url]).returncode
    assert rc == 0 and os.path.getsize(dest + '.part') >= mn, f'download failed: {fn}'
    os.replace(dest + '.part', dest)
    print('downloaded', fn)
print('BART snapshot ready')
print('Output directory:', OUTPUT_DIR)


## 4. Extract frozen artifacts

In [ ]:
def extract_archive_once(archive_path, target_dir, required_name):
    required_path = os.path.join(target_dir, required_name)
    if os.path.isfile(required_path):
        print('Already extracted:', target_dir)
        return required_path

    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(archive_path, 'r') as archive:
        archive.extractall(target_dir)

    assert os.path.isfile(required_path), (
        f'{required_name} not found after extracting {archive_path}'
    )
    print('Extracted:', archive_path, '->', target_dir)
    return required_path


DUAL_EPOCH2_PATH = extract_archive_once(
    OPERATING_POINT_ARCHIVE,
    OPERATING_POINT_EXTRACT_DIR,
    'checkpoints/dual_epoch_2.pt',
)
SELECTED_OPERATING_POINT_PATH = os.path.join(
    OPERATING_POINT_EXTRACT_DIR,
    'selected_operating_point.json',
)

PREVIOUS_EPOCH1_PRED_PATH = extract_archive_once(
    PREVIOUS_DUAL_TEST_ARCHIVE,
    PREVIOUS_DUAL_TEST_DIR,
    f'test_preds_dual_head_selector_{EXPECTED_EPOCH1_TEST_SHA_PREFIX}.json',
)
PREVIOUS_EPOCH1_HARD_PRED_PATH = os.path.join(
    PREVIOUS_DUAL_TEST_DIR,
    f'test_preds_dual_head_selector_hard_v2_{EXPECTED_EPOCH1_TEST_SHA_PREFIX}.json',
)
PREVIOUS_EPOCH1_TRIGGER_PATH = os.path.join(
    PREVIOUS_DUAL_TEST_DIR,
    f'test_triggers_dual_head_selector_{EXPECTED_EPOCH1_TEST_SHA_PREFIX}.json',
)
PREVIOUS_EPOCH1_HARD_TRIGGER_PATH = os.path.join(
    PREVIOUS_DUAL_TEST_DIR,
    f'test_triggers_dual_head_selector_hard_v2_{EXPECTED_EPOCH1_TEST_SHA_PREFIX}.json',
)

SHARED_PRED_PATH = extract_archive_once(
    SELECTOR_TEST_ARCHIVE,
    SELECTOR_TEST_DIR,
    'preds_fusion_selector_only.json',
)
SHARED_HARD_PRED_PATH = os.path.join(
    SELECTOR_TEST_DIR,
    'preds_fusion_selector_hard_v2.json',
)
SHARED_TRIGGER_PATH = os.path.join(
    SELECTOR_TEST_DIR,
    'triggers_fusion_selector_only.json',
)
SHARED_HARD_TRIGGER_PATH = os.path.join(
    SELECTOR_TEST_DIR,
    'triggers_fusion_selector_hard_v2.json',
)

for path in [
    DUAL_EPOCH2_PATH,
    SELECTED_OPERATING_POINT_PATH,
    PREVIOUS_EPOCH1_PRED_PATH,
    PREVIOUS_EPOCH1_HARD_PRED_PATH,
    PREVIOUS_EPOCH1_TRIGGER_PATH,
    PREVIOUS_EPOCH1_HARD_TRIGGER_PATH,
    SHARED_PRED_PATH,
    SHARED_HARD_PRED_PATH,
    SHARED_TRIGGER_PATH,
    SHARED_HARD_TRIGGER_PATH,
]:
    assert os.path.isfile(path), f'Missing extracted file: {path}'

with open(SELECTED_OPERATING_POINT_PATH, encoding='utf-8') as handle:
    selected_operating_point = json.load(handle)

selected_dual = selected_operating_point['selected_dual']
assert selected_dual['model_key'] == 'dual_e2', selected_dual
assert abs(float(selected_dual['tau']) - SELECTOR_TAU) < 1e-12, selected_dual

print('Epoch-2 checkpoint:', DUAL_EPOCH2_PATH)
print('Development-selected operating point:', selected_dual)


## 5. Load the fusion backbone

In [ ]:
# 3. Load data + frozen fusion model.
from transformers import BartTokenizer, BartForConditionalGeneration
tokenizer = BartTokenizer.from_pretrained(BART_LOCAL)
bart = BartForConditionalGeneration.from_pretrained(BART_LOCAL, local_files_only=True).to(DEVICE)
data, graphs, vocab = load_artifacts(PROCESSED_DIR)
num_relations = len(vocab['relation_vocab']) * 2
model = FusionOnlyGNNModel(bart, num_relations=num_relations).to(DEVICE)
model = load_variant_checkpoint_resume(model, CKPT_FUSION, DEVICE, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print('frozen fusion model ready | splits:', {k: len(v) for k, v in data.items()})

In [ ]:
fusion_model = model
fusion_model.eval()
for p in fusion_model.parameters(): p.requires_grad_(False)
print('Frozen fusion backbone registered as fusion_model.')


## 6. Rebuild repaired-test manifest and metrics

In [ ]:
# 4. Manifest + metrics + TrieMapV2 + hard mask (ports of the 12b-verified cells).
from collections import OrderedDict
from difflib import SequenceMatcher
import sacrebleu

def _tp(t):
    if isinstance(t, dict): return str(t.get('subject','')), str(t.get('predicate','')), str(t.get('object',''))
    return str(t[0]), str(t[1]), str(t[2])

groups = OrderedDict()
for i, ex in enumerate(data['test']):
    key = json.dumps([_tp(t) for t in ex['triples']], ensure_ascii=False)
    g = groups.setdefault(key, {'first_index': i, 'references': []})
    for v in list(ex.get('all_targets') or []) + [ex.get('target','')]:
        v = str(v).strip()
        if v and v not in g['references']: g['references'].append(v)
train_predicates = {_tp(t)[1] for ex in data['train'] for t in ex['triples']}
ITEMS = []
for uid, (key, g) in enumerate(groups.items()):
    ex = dict(data['test'][g['first_index']]); ex['all_targets'] = g['references']
    ITEMS.append({'uid': uid, 'idx': g['first_index'], 'ex': ex,
                  'unseen': bool({_tp(t)[1] for t in ex['triples']} - train_predicates)})
assert len(ITEMS) == 2510 and sum(x['unseen'] for x in ITEMS) == 752

_TR = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
       'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())
def _sa(s):
    s = ''.join(_TR.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
def _norm(s):
    s = _sa(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)
def _wm(t, s):
    return bool(s) and re.search(r'(?<![a-z0-9])' + re.escape(s) + r'(?![a-z0-9])', t) is not None
def _dst(v):
    toks = set(); raw = _sa(str(v)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups()); toks.add(str(y))
        if 1 <= mo <= 12: toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{sx}' for sx in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw): toks.add(raw)
    return toks
def _dg(s): return re.sub(r'[^0-9]', '', str(s))
def grounding_score(prediction, triples):
    pn = _norm(prediction)
    forms, tokens, num = [], set(), set()
    for t in triples:
        s, p, o = _tp(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_dst(v)); d = _dg(v)
            if d: num.add(d)
        tokens.update(_norm(re.sub(r'([a-z])([A-Z])', r'\1 \2', p)).split())
    fd = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _wm(pn, f) or all(_wm(pn, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _sa(prediction)):
        n = _norm(m)
        if len(n) >= 3 and (' ' in n or n not in _STOP): men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(prediction)):
        if any(c.isdigit() for c in m): men.add(_norm(m))
    men = {m for m in men if m}
    def ok(m):
        for f in forms:
            if m == f or _wm(f, m): return True
        if all(t in tokens for t in m.split()): return True
        d = _dg(m)
        if d and any(d in c or c in d for c in num): return True
        md = m.replace(' ', '')
        return len(md) >= 3 and any(md in x or x in md for x in fd)
    hall = sorted(m for m in men if not ok(m))
    corr = [m for m in hall if max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0) >= 0.55]
    return {'halluc': len(hall)/len(men) if men else 0.0, 'recall': recall,
            'hallucinated': hall, 'corruptions': corr}
ART = {'iso': r'[A-Za-z],? \d{3,4}-\d{1,2}-\d{1,2}|\d{1,2}(st|nd|rd|th)? [A-Za-z]+ \d{3,4}-\d{1,2}-\d{1,2}',
       'paren': r'\((The [^)]+album|[0-9]{4} film|film|band|song|album|actor[^)]*|footballer[^)]*|musician[^)]*)\)',
       'unit': r'\d[\d.,]*\s*\((milli|centi|kilo)?(metres|meters|grams|litres|liters|inches)\)'}
def artrow(p): return any(re.search(x, str(p)) for x in ART.values())
def corpus_bleu_lc(preds, refs_list):
    maxr = max(len(r) for r in refs_list)
    streams = [[r[k] if k < len(r) else r[0] for r in refs_list] for k in range(maxr)]
    return sacrebleu.corpus_bleu(preds, streams, lowercase=True, tokenize='13a').score

class TNode:
    __slots__ = ('ch', 'score', 'mx', 'nterm', 'terminal')
    def __init__(self):
        self.ch = {}; self.score = None; self.mx = -1e9; self.nterm = 0; self.terminal = False
_LIT = [r'^[\d\s.,:/\-+%°"]*$', r'^\d{3,4}-\d{1,2}-\d{1,2}', r'^\d+(\.\d+)?$']
def is_literal(name):
    n = str(name).strip().strip('"').strip("'").strip()
    return len(n) < 2 or any(re.match(p, n) for p in _LIT)
def clean_surface(name):
    s = str(name).strip().strip('"').strip("'").replace('_', ' ')
    s = re.sub(r'\s*\([^)]*\)', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
class TrieMapV2:
    def __init__(self, entity_names, tokenizer):
        self.root = TNode(); self.kept = {}
        for ni, name in enumerate(entity_names):
            if is_literal(name): continue
            base = clean_surface(name)
            if not base or is_literal(base): continue
            self.kept[ni] = base
            for v in (base, ' ' + base):
                ids = tokenizer.encode(v, add_special_tokens=False)
                if not ids: continue
                n = self.root
                for tid in ids: n = n.ch.setdefault(int(tid), TNode())
                n.terminal = True
        self._cache(self.root)
    def _cache(self, n):
        nt = 1 if n.terminal else 0
        for c in n.ch.values():
            self._cache(c); nt += c.nterm
        n.nterm = nt
    def suffix_matches(self, gen_ids, lookback=20):
        out = []
        for start in range(max(0, len(gen_ids) - lookback), len(gen_ids)):
            n = self.root; okk = True
            for pos in range(start, len(gen_ids)):
                t = int(gen_ids[pos])
                if t not in n.ch: okk = False; break
                n = n.ch[t]
            if okk and n is not self.root: out.append((len(gen_ids) - start, n))
        return out
def hard_mask_v2(logits, trie, gen_ids):
    best = None
    for depth, node in trie.suffix_matches(gen_ids):
        if node.terminal or not node.ch: continue
        if (depth >= LOCK_MIN_DEPTH or node.nterm == 1) and (best is None or depth > best[0]):
            best = (depth, node)
    if best is None: return logits, False
    legal = list(best[1].ch.keys())
    if float(logits.max()) - max(float(logits[t]) for t in legal) > ESCAPE_MARGIN:
        return logits, False
    m = torch.full_like(logits, -1e9); m[legal] = 0.0
    return logits + m, True
print('protocol cells ready')

In [ ]:
# Protocol checksum for the repaired-test ordering.
REPAIRED_ORDER_PATH = os.path.join(OUTPUT_DIR, 'repaired_test_order.json')
repaired_order = [
    {
        'uid': item['uid'],
        'raw_test_index': item['idx'],
        'unseen': item['unseen'],
        'triples': [_tp(t) for t in item['ex']['triples']],
    }
    for item in ITEMS
]
atomic_order_payload = json.dumps(
    repaired_order,
    ensure_ascii=False,
    sort_keys=True,
    separators=(',', ':'),
).encode('utf-8')
REPAIRED_ORDER_SHA = hashlib.sha256(atomic_order_payload).hexdigest()
with open(REPAIRED_ORDER_PATH, 'w', encoding='utf-8') as handle:
    json.dump(repaired_order, handle, indent=2, ensure_ascii=False)

print('Repaired inputs:', len(ITEMS))
print('Seen:', sum(not item['unseen'] for item in ITEMS))
print('Unseen:', sum(item['unseen'] for item in ITEMS))
print('Ordering SHA256:', REPAIRED_ORDER_SHA)


## 7. Load shared and dual selectors

In [ ]:
# 5. Selector module + gold span labels from references.
class EntitySelector(nn.Module):
    # scores {NONE} ∪ {entities} from [h_t ; e_i ; cov_i]
    def __init__(self, d=768, hid=256):
        super().__init__()
        self.ent_mlp = nn.Sequential(nn.Linear(2*d + 1, hid), nn.ReLU(), nn.Linear(hid, 1))
        self.none_mlp = nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, h, ents, cov):
        # h: (L,d) | ents: (N,d) | cov: (L,N) in {0,1}
        L, d = h.shape; N = ents.shape[0]
        he = torch.cat([h.unsqueeze(1).expand(L, N, d), ents.unsqueeze(0).expand(L, N, d),
                        cov.unsqueeze(-1)], dim=-1)
        s_ent = self.ent_mlp(he).squeeze(-1)          # (L,N)
        s_none = self.none_mlp(h)                     # (L,1)
        return torch.cat([s_none, s_ent], dim=-1)     # (L, 1+N); class 0 = NONE

def target_labels(ex, graph, target_ids):
    # label per decoder-state position t (predicting token t of target_ids):
    #  0 = NONE, i+1 = entity i STARTS at t, -100 = inside a span (ignored)
    names = list(getattr(graph, 'entity_names', []) or [])
    surfaces = [clean_surface(n) if not is_literal(n) else '' for n in names]
    lab = np.zeros(len(target_ids), dtype=np.int64)
    text = ex['target']
    spans = find_entity_token_spans(text, [s if s else '§none§' for s in surfaces], tokenizer)
    for ni, (s, e) in enumerate(spans):
        if s < 0 or not surfaces[ni]: continue
        if s < len(lab): lab[s] = ni + 1
        for k in range(s + 1, min(e, len(lab))): lab[k] = -100
    return lab
print('selector defined')

In [ ]:
from torch_geometric.nn import RGCNConv

class ResidualConstraintRGCNHead(nn.Module):
    """One relation-aware constraint head with function-preserving initialization."""
    def __init__(self, d=768, bottleneck=256, num_relations=None, num_bases=8, dropout=0.1):
        super().__init__()
        assert num_relations is not None
        self.conv = RGCNConv(d, bottleneck, num_relations, num_bases=min(num_bases, num_relations))
        self.norm = nn.LayerNorm(bottleneck)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(bottleneck, d)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.residual_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, g_fusion, edge_index, edge_type):
        h = self.conv(g_fusion, edge_index, edge_type)
        h = self.dropout(self.norm(F.gelu(h)))
        delta = self.out(h)
        g_constraint = g_fusion + self.residual_scale * delta
        return g_constraint, delta

class DualHeadAdaptiveGNN(nn.Module):
    def __init__(self, d, bottleneck, num_relations, num_bases, dropout):
        super().__init__()
        self.constraint_head = ResidualConstraintRGCNHead(
            d=d, bottleneck=bottleneck, num_relations=num_relations,
            num_bases=num_bases, dropout=dropout,
        )
        self.selector = EntitySelector(d=d, hid=256)

    def constraint_states(self, g_fusion, edge_index, edge_type):
        return self.constraint_head(g_fusion, edge_index, edge_type)

def nparams(module, trainable=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable))

print('Dual-head architecture ready.')


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json_dump(value, path, indent=2):
    temporary = path + '.tmp'
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=indent, ensure_ascii=False)
    os.replace(temporary, path)


current_checkpoint = torch.load(CURRENT_SELECTOR_CKPT, map_location=DEVICE)
current_selector = EntitySelector(d=MODEL_DIM, hid=256).to(DEVICE)
current_selector.load_state_dict(current_checkpoint['state'])
current_selector.eval()
for parameter in current_selector.parameters():
    parameter.requires_grad_(False)


dual_model = DualHeadAdaptiveGNN(
    d=MODEL_DIM,
    bottleneck=CONSTRAINT_BOTTLENECK,
    num_relations=num_relations,
    num_bases=CONSTRAINT_NUM_BASES,
    dropout=CONSTRAINT_DROPOUT,
).to(DEVICE)

dual_checkpoint = torch.load(DUAL_EPOCH2_PATH, map_location=DEVICE)
dual_model.load_state_dict(dual_checkpoint['state'])
dual_model.eval()
for parameter in dual_model.parameters():
    parameter.requires_grad_(False)

DUAL_EPOCH = int(dual_checkpoint['epoch'])
DUAL_EPOCH2_SHA = sha256_file(DUAL_EPOCH2_PATH)
CURRENT_SELECTOR_SHA = sha256_file(CURRENT_SELECTOR_CKPT)

print('Current selector epoch:', current_checkpoint.get('epoch'))
print('Current selector SHA256:', CURRENT_SELECTOR_SHA)
print('Dual-head selected epoch:', DUAL_EPOCH)
print('Dual-head epoch-2 SHA256:', DUAL_EPOCH2_SHA)
print('Residual scale:', float(dual_model.constraint_head.residual_scale.detach().cpu()))

assert DUAL_EPOCH == 2, f'Expected epoch 2, found epoch {DUAL_EPOCH}'
assert DUAL_EPOCH2_SHA == EXPECTED_EPOCH2_SHA, (
    f'Unexpected epoch-2 SHA: {DUAL_EPOCH2_SHA}'
)


## 8. Adaptive decoding and Hard-v2

In [ ]:
@torch.no_grad()
def decode_adaptive(ex, graph, mode='dual', use_hard=True, tau=SELECTOR_TAU, margin=COMPATIBILITY_MARGIN, return_events=False):
    assert mode in ('shared', 'dual')
    enc_in = tokenizer(ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
                       padding='max_length', return_tensors='pt')
    inp, att = enc_in['input_ids'].to(DEVICE), enc_in['attention_mask'].to(DEVICE)
    gb = torch.zeros(graph.x.size(0), dtype=torch.long, device=DEVICE)
    enc = fusion_model.bart.model.encoder(input_ids=inp, attention_mask=att)
    edge_index, edge_type = graph.edge_index.to(DEVICE), graph.edge_type.to(DEVICE)
    g_fusion, _ = fusion_model.rgcn(graph.x.to(DEVICE), edge_index, edge_type)
    h_pad, k_mask = pad_kg_nodes(g_fusion, gb, 1)
    if mode == 'dual':
        selector_ents, _ = dual_model.constraint_states(g_fusion.float(), edge_index, edge_type)
        selector_obj = dual_model.selector
    else:
        selector_ents = g_fusion.float(); selector_obj = current_selector
    names = list(getattr(graph, 'entity_names', []) or [])
    trie = TrieMapV2(names, tokenizer)
    ent_ids = {}
    for ni, base in trie.kept.items():
        a = tokenizer.encode(base, add_special_tokens=False)
        b = tokenizer.encode(' ' + base, add_special_tokens=False)
        if a and b: ent_ids[ni] = (a, b)
    start_id, eos = fusion_model.bart.config.decoder_start_token_id, fusion_model.bart.config.eos_token_id
    gen = [start_id]; past = None; committed = None; selector_triggers = 0; hard_triggers = 0; events = []
    for step in range(MAX_GEN_LEN - 1):
        di = torch.tensor([[gen[-1]]], dtype=torch.long, device=DEVICE)
        out = fusion_model.bart.model.decoder(
            input_ids=di, encoder_hidden_states=enc.last_hidden_state,
            encoder_attention_mask=att, past_key_values=past, use_cache=True)
        past = out.past_key_values
        h, _, _ = fusion_model.kg_cross_attention(out.last_hidden_state, h_pad, k_mask)
        logits = add_final_logits_bias(fusion_model.bart, fusion_model.bart.lm_head(h))[:, -1, :].squeeze(0).float().cpu()
        if committed is not None:
            nxt = committed['ids'][committed['pos']]; committed['pos'] += 1
            if committed['pos'] >= len(committed['ids']): committed = None
        else:
            if use_hard:
                logits2, active = hard_mask_v2(logits, trie, gen[1:]); hard_triggers += int(active)
            else: logits2 = logits
            nxt = int(logits2.argmax())
            if ent_ids:
                decoded = tokenizer.decode(gen[1:], skip_special_tokens=True); dn = _norm(decoded)
                cov_flags = [1.0 if (trie.kept.get(i) and _wm(dn, _norm(trie.kept[i]))) else 0.0
                             for i in range(len(names))]
                cov = torch.tensor([cov_flags], device=DEVICE)
                sel = selector_obj(h[:, -1, :].float(), selector_ents.float(), cov).squeeze(0)
                ent_logits = sel[1:].clone()
                for i in range(len(names)):
                    if i not in ent_ids or cov_flags[i] > 0: ent_logits[i] = -1e9
                probs = torch.softmax(torch.cat([sel[:1], ent_logits]), -1)
                if probs.numel() > 1:
                    best_i = int(probs[1:].argmax()); p_best = float(probs[1+best_i])
                    if p_best >= tau and best_i in ent_ids:
                        seq = ent_ids[best_i][0] if len(gen) == 1 else ent_ids[best_i][1]
                        gap = float(logits.max()) - float(logits[seq[0]])
                        if gap <= margin:
                            nxt = seq[0]; selector_triggers += 1
                            if len(seq) > 1: committed = {'ids': seq, 'pos': 1}
                            events.append({'step': step, 'entity_index': best_i,
                                           'entity': trie.kept.get(best_i, names[best_i]),
                                           'probability': p_best, 'bart_first_token_gap': gap})
        if nxt == eos: break
        gen.append(nxt)
    result = {'prediction': tokenizer.decode(gen[1:], skip_special_tokens=True),
              'selector_triggers': selector_triggers, 'hard_v2_triggers': hard_triggers}
    if return_events: result['events'] = events
    return result

print('Decoder ready. TrieMap is used only by Hard-v2; selector commitment uses the chosen entity token sequence directly.')


## 9. Load and validate the four existing arms

In [ ]:
def load_json_list(path):
    with open(path, encoding='utf-8') as handle:
        value = json.load(handle)
    assert isinstance(value, list), f'Expected a list: {path}'
    return value


def candidate_files(filename, preferred_paths):
    candidates = []
    for path in preferred_paths:
        if path and os.path.isfile(path):
            candidates.append(path)
    for root, _, files in os.walk(PROJECT_DIR):
        if filename in files:
            path = os.path.join(root, filename)
            if path not in candidates:
                candidates.append(path)
    return candidates


def resolve_2510_prediction_file(filename, preferred_paths):
    valid = []
    for path in candidate_files(filename, preferred_paths):
        try:
            values = load_json_list(path)
        except Exception as error:
            print('Skipping unreadable candidate:', path, error)
            continue
        if len(values) == len(ITEMS):
            valid.append((path, values))
        else:
            print('Skipping wrong-length candidate:', path, len(values))

    assert valid, (
        f'No {len(ITEMS)}-prediction file found for {filename}. '
        'Place the established hard-v2 evaluation files in PROJECT_DIR.'
    )
    path, values = valid[0]
    print('Resolved', filename, '->', path)
    return path, values


# Explicit preferred locations from the established Hard-v2 run.
fusion_path, fusion_only = resolve_2510_prediction_file(
    'preds_fusion_off.json',
    [
        f'{PROJECT_DIR}/hard_v2_eval_fixed_v4/preds_fusion_off.json',
        f'{PROJECT_DIR}/preds_fusion_off.json',
    ],
)
fusion_hard_path, fusion_hard = resolve_2510_prediction_file(
    'preds_fusion_hard_v2.json',
    [
        f'{PROJECT_DIR}/hard_v2_eval_fixed_v4/preds_fusion_hard_v2.json',
        f'{PROJECT_DIR}/preds_fusion_hard_v2.json',
    ],
)

shared_selector = load_json_list(SHARED_PRED_PATH)
shared_selector_hard = load_json_list(SHARED_HARD_PRED_PATH)
shared_triggers = load_json_list(SHARED_TRIGGER_PATH)
shared_hard_triggers = load_json_list(SHARED_HARD_TRIGGER_PATH)

for name, predictions in {
    'fusion_only': fusion_only,
    'fusion_hard_v2': fusion_hard,
    'shared_selector': shared_selector,
    'shared_selector_hard_v2': shared_selector_hard,
}.items():
    assert len(predictions) == len(ITEMS), (
        f'{name}: expected {len(ITEMS)} predictions, found {len(predictions)}'
    )


def test_block(predictions, indices):
    selected_predictions = [predictions[index] for index in indices]
    selected_items = [ITEMS[index] for index in indices]
    scores = [
        grounding_score(prediction, item['ex']['triples'])
        for prediction, item in zip(selected_predictions, selected_items)
    ]
    return {
        'n': len(indices),
        'bleu': corpus_bleu_lc(
            selected_predictions,
            [item['ex']['all_targets'] for item in selected_items],
        ),
        'halluc': float(np.mean([score['halluc'] for score in scores])),
        'recall': float(np.mean([score['recall'] for score in scores])),
        'corr_rows': int(sum(bool(score['corruptions']) for score in scores)),
        'art_rows': int(sum(artrow(prediction) for prediction in selected_predictions)),
    }


# Reproduce established metrics before generating a new arm. This catches
# wrong files, wrong ordering, or a mismatched scoring implementation.
overall_indices = list(range(len(ITEMS)))
existing_metrics = {
    'fusion_only': test_block(fusion_only, overall_indices),
    'fusion_hard_v2': test_block(fusion_hard, overall_indices),
    'shared_selector': test_block(shared_selector, overall_indices),
    'shared_selector_hard_v2': test_block(shared_selector_hard, overall_indices),
}
display(pd.DataFrame([
    {'arm': name, **metrics}
    for name, metrics in existing_metrics.items()
]))

expected = {
    'fusion_only': {'bleu': 47.318, 'halluc': 0.03750, 'recall': 0.79992},
    'fusion_hard_v2': {'bleu': 47.681, 'halluc': 0.01885, 'recall': 0.85175},
    'shared_selector': {'bleu': 48.716, 'halluc': 0.01857, 'recall': 0.84213},
    'shared_selector_hard_v2': {'bleu': 48.863, 'halluc': 0.01255, 'recall': 0.86487},
}
for arm_name, target in expected.items():
    observed = existing_metrics[arm_name]
    assert abs(observed['bleu'] - target['bleu']) < 0.08, (arm_name, observed, target)
    assert abs(observed['halluc'] - target['halluc']) < 0.0015, (arm_name, observed, target)
    assert abs(observed['recall'] - target['recall']) < 0.0015, (arm_name, observed, target)

print('PASS: all four reused arms match the established repaired-test scores.')


## 10. Generate the two frozen dual-head arms

In [ ]:
def run_dual_test_arm(tag, use_hard):
    cache_tag = f'{tag}_{DUAL_EPOCH2_SHA[:12]}'
    prediction_path = os.path.join(OUTPUT_DIR, f'test_preds_{cache_tag}.json')
    trigger_path = os.path.join(OUTPUT_DIR, f'test_triggers_{cache_tag}.json')
    event_path = os.path.join(OUTPUT_DIR, f'test_events_{cache_tag}.json')

    predictions = load_json_list(prediction_path) if os.path.exists(prediction_path) else []
    triggers = load_json_list(trigger_path) if os.path.exists(trigger_path) else []
    events = load_json_list(event_path) if os.path.exists(event_path) else []

    assert len(predictions) == len(triggers) == len(events)
    assert len(predictions) <= len(ITEMS)

    started = time.time()
    for item_index in range(len(predictions), len(ITEMS)):
        item = ITEMS[item_index]
        result = decode_adaptive(
            item['ex'],
            graphs['test'][item['idx']],
            mode='dual',
            use_hard=use_hard,
            tau=SELECTOR_TAU,
            margin=COMPATIBILITY_MARGIN,
            return_events=True,
        )
        predictions.append(result['prediction'])
        triggers.append({
            'selector': result['selector_triggers'],
            'hard_v2': result['hard_v2_triggers'],
        })
        events.append(result.get('events', []))

        if (item_index + 1) % SAVE_EVERY == 0 or item_index + 1 == len(ITEMS):
            atomic_json_dump(predictions, prediction_path)
            atomic_json_dump(triggers, trigger_path)
            atomic_json_dump(events, event_path)

        if (item_index + 1) % 250 == 0:
            print(
                f'[{tag}] {item_index + 1}/{len(ITEMS)} '
                f'({time.time() - started:.0f}s)'
            )

    return predictions, triggers, events


# Reuse frozen epoch-1 repaired-test outputs.
dual_epoch1 = load_json_list(PREVIOUS_EPOCH1_PRED_PATH)
dual_epoch1_hard = load_json_list(PREVIOUS_EPOCH1_HARD_PRED_PATH)
dual_epoch1_triggers = load_json_list(PREVIOUS_EPOCH1_TRIGGER_PATH)
dual_epoch1_hard_triggers = load_json_list(PREVIOUS_EPOCH1_HARD_TRIGGER_PATH)

# Generate only the two new epoch-2 arms.
dual_epoch2, dual_epoch2_triggers, dual_epoch2_events = run_dual_test_arm(
    'dual_head_epoch2',
    use_hard=False,
)
dual_epoch2_hard, dual_epoch2_hard_triggers, dual_epoch2_hard_events = run_dual_test_arm(
    'dual_head_epoch2_hard_v2',
    use_hard=True,
)

for name, values in {
    'dual_epoch1': dual_epoch1,
    'dual_epoch1_hard': dual_epoch1_hard,
    'dual_epoch2': dual_epoch2,
    'dual_epoch2_hard': dual_epoch2_hard,
}.items():
    assert len(values) == len(ITEMS), f'{name}: {len(values)} != {len(ITEMS)}'

print('Epoch-2 repaired-test generation complete.')


## 11. Overall, seen, and unseen metrics

In [ ]:
ARMS = {
    'fusion_only': fusion_only,
    'shared_selector': shared_selector,
    'dual_epoch1': dual_epoch1,
    'dual_epoch2': dual_epoch2,
    'fusion_hard_v2': fusion_hard,
    'shared_selector_hard_v2': shared_selector_hard,
    'dual_epoch1_hard_v2': dual_epoch1_hard,
    'dual_epoch2_hard_v2': dual_epoch2_hard,
}

SUBSETS = {
    'overall': list(range(len(ITEMS))),
    'seen': [index for index, item in enumerate(ITEMS) if not item['unseen']],
    'unseen': [index for index, item in enumerate(ITEMS) if item['unseen']],
}

TEST_SUMMARY = {
    'status': 'frozen post-development descriptive evaluation; not pristine confirmatory',
    'repaired_test_order_sha256': REPAIRED_ORDER_SHA,
    'checkpoint': {
        'path': DUAL_EPOCH2_PATH,
        'sha256': DUAL_EPOCH2_SHA,
        'epoch': DUAL_EPOCH,
        'selected_on': 'development-only trigger-budget matching in notebook 19',
    },
    'frozen_decode': {
        'selector_tau': SELECTOR_TAU,
        'compatibility_margin': COMPATIBILITY_MARGIN,
        'hard_v2_min_depth': LOCK_MIN_DEPTH,
        'hard_v2_escape_margin': ESCAPE_MARGIN,
    },
    'arms': {
        arm_name: {
            subset_name: test_block(predictions, indices)
            for subset_name, indices in SUBSETS.items()
        }
        for arm_name, predictions in ARMS.items()
    },
}

summary_rows = [
    {'arm': arm_name, 'subset': subset_name, **metrics}
    for arm_name, arm_results in TEST_SUMMARY['arms'].items()
    for subset_name, metrics in arm_results.items()
]
summary_frame = pd.DataFrame(summary_rows)
display(summary_frame)

atomic_json_dump(
    TEST_SUMMARY,
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_summary.json'),
)
summary_frame.to_csv(
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_summary.csv'),
    index=False,
)

comparison_pairs = {
    'shared_vs_fusion': ('shared_selector', 'fusion_only'),
    'dual_e1_vs_fusion': ('dual_epoch1', 'fusion_only'),
    'dual_e2_vs_fusion': ('dual_epoch2', 'fusion_only'),
    'dual_e1_vs_shared': ('dual_epoch1', 'shared_selector'),
    'dual_e2_vs_shared': ('dual_epoch2', 'shared_selector'),
    'dual_e2_vs_dual_e1': ('dual_epoch2', 'dual_epoch1'),
    'shared_hard_vs_fusion_hard': ('shared_selector_hard_v2', 'fusion_hard_v2'),
    'dual_e1_hard_vs_fusion_hard': ('dual_epoch1_hard_v2', 'fusion_hard_v2'),
    'dual_e2_hard_vs_fusion_hard': ('dual_epoch2_hard_v2', 'fusion_hard_v2'),
    'dual_e1_hard_vs_shared_hard': ('dual_epoch1_hard_v2', 'shared_selector_hard_v2'),
    'dual_e2_hard_vs_shared_hard': ('dual_epoch2_hard_v2', 'shared_selector_hard_v2'),
    'dual_e2_hard_vs_dual_e1_hard': ('dual_epoch2_hard_v2', 'dual_epoch1_hard_v2'),
}

delta_rows = []
for comparison_name, (candidate_name, baseline_name) in comparison_pairs.items():
    for subset_name in SUBSETS:
        candidate = TEST_SUMMARY['arms'][candidate_name][subset_name]
        baseline = TEST_SUMMARY['arms'][baseline_name][subset_name]
        delta_rows.append({
            'comparison': comparison_name,
            'subset': subset_name,
            'BLEU_delta': candidate['bleu'] - baseline['bleu'],
            'recall_delta_pp': 100 * (candidate['recall'] - baseline['recall']),
            'halluc_delta_pp': 100 * (candidate['halluc'] - baseline['halluc']),
            'corruption_row_delta': candidate['corr_rows'] - baseline['corr_rows'],
            'artifact_row_delta': candidate['art_rows'] - baseline['art_rows'],
        })

delta_frame = pd.DataFrame(delta_rows)
display(delta_frame)
delta_frame.to_csv(
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_deltas.csv'),
    index=False,
)


## 12. Paired bootstrap comparisons

In [ ]:
def per_example_values(predictions, indices, metric):
    return np.array([
        grounding_score(predictions[index], ITEMS[index]['ex']['triples'])[metric]
        for index in indices
    ], dtype=float)


def paired_bootstrap_metric(
    candidate_predictions,
    baseline_predictions,
    indices,
    metric,
    samples=BOOTSTRAP_SAMPLES,
    seed=20260725,
):
    candidate_values = per_example_values(candidate_predictions, indices, metric)
    baseline_values = per_example_values(baseline_predictions, indices, metric)
    differences = candidate_values - baseline_values

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(samples, dtype=float)
    for sample_index in range(samples):
        sampled = rng.integers(0, len(differences), len(differences))
        bootstrap_means[sample_index] = differences[sampled].mean()

    if metric == 'recall':
        improved = int((differences > 0).sum())
        worsened = int((differences < 0).sum())
    else:
        improved = int((differences < 0).sum())
        worsened = int((differences > 0).sum())

    return {
        'delta': float(differences.mean()),
        'ci95': [
            float(np.percentile(bootstrap_means, 2.5)),
            float(np.percentile(bootstrap_means, 97.5)),
        ],
        'improved': improved,
        'worsened': worsened,
        'tied': int((differences == 0).sum()),
    }


def paired_bleu_bootstrap(
    candidate_predictions,
    baseline_predictions,
    indices,
    samples=BLEU_BOOTSTRAP_SAMPLES,
    seed=20260725,
):
    rng = np.random.default_rng(seed)
    deltas = np.empty(samples, dtype=float)
    references = [ITEMS[index]['ex']['all_targets'] for index in indices]
    candidate = [candidate_predictions[index] for index in indices]
    baseline = [baseline_predictions[index] for index in indices]

    for sample_index in range(samples):
        sampled = rng.integers(0, len(indices), len(indices))
        candidate_sample = [candidate[index] for index in sampled]
        baseline_sample = [baseline[index] for index in sampled]
        reference_sample = [references[index] for index in sampled]
        deltas[sample_index] = (
            corpus_bleu_lc(candidate_sample, reference_sample)
            - corpus_bleu_lc(baseline_sample, reference_sample)
        )

    observed = (
        corpus_bleu_lc(candidate, references)
        - corpus_bleu_lc(baseline, references)
    )
    return {
        'delta': float(observed),
        'ci95': [
            float(np.percentile(deltas, 2.5)),
            float(np.percentile(deltas, 97.5)),
        ],
    }


PAIRED_RESULTS = {}
paired_rows = []
for comparison_name, (candidate_name, baseline_name) in comparison_pairs.items():
    candidate_predictions = ARMS[candidate_name]
    baseline_predictions = ARMS[baseline_name]
    PAIRED_RESULTS[comparison_name] = {}

    for subset_name, indices in SUBSETS.items():
        subset_result = {
            metric: paired_bootstrap_metric(
                candidate_predictions,
                baseline_predictions,
                indices,
                metric,
                samples=BOOTSTRAP_SAMPLES,
                seed=20260725 + len(indices),
            )
            for metric in ('recall', 'halluc')
        }
        subset_result['changed_outputs'] = int(sum(
            candidate_predictions[index] != baseline_predictions[index]
            for index in indices
        ))

        if RUN_OPTIONAL_BLEU_BOOTSTRAP:
            subset_result['bleu'] = paired_bleu_bootstrap(
                candidate_predictions,
                baseline_predictions,
                indices,
            )

        PAIRED_RESULTS[comparison_name][subset_name] = subset_result

        for metric in ('recall', 'halluc'):
            result = subset_result[metric]
            paired_rows.append({
                'comparison': comparison_name,
                'subset': subset_name,
                'metric': metric,
                'delta': result['delta'],
                'ci_low': result['ci95'][0],
                'ci_high': result['ci95'][1],
                'improved': result['improved'],
                'worsened': result['worsened'],
                'tied': result['tied'],
                'changed_outputs': subset_result['changed_outputs'],
            })

paired_frame = pd.DataFrame(paired_rows)
display(paired_frame)

atomic_json_dump(
    PAIRED_RESULTS,
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_paired.json'),
)
paired_frame.to_csv(
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_paired.csv'),
    index=False,
)


## 13. Trigger analysis

In [ ]:
def trigger_summary(rows, name):
    if not rows:
        return {
            'arm': name,
            'examples': 0,
            'selector_triggers': 0,
            'examples_with_selector_trigger': 0,
            'mean_selector_triggers': 0.0,
            'hard_v2_counts_available': False,
            'hard_v2_triggers': None,
            'examples_with_hard_v2_trigger': None,
            'mean_hard_v2_triggers': None,
        }

    first = rows[0]
    if isinstance(first, dict):
        selector_counts = np.asarray(
            [int(row.get('selector', 0)) for row in rows], dtype=np.int64)
        hard_counts = np.asarray(
            [int(row.get('hard_v2', 0)) for row in rows], dtype=np.int64)
        hard_available = True
    elif isinstance(first, (int, float, np.integer, np.floating)):
        selector_counts = np.asarray(rows, dtype=np.int64)
        hard_counts = None
        hard_available = False
    else:
        raise TypeError(
            f'Unsupported trigger-row type for {name}: {type(first).__name__}')

    result = {
        'arm': name,
        'examples': len(rows),
        'selector_triggers': int(selector_counts.sum()),
        'examples_with_selector_trigger': int((selector_counts > 0).sum()),
        'mean_selector_triggers': float(selector_counts.mean()),
        'hard_v2_counts_available': hard_available,
    }
    if hard_available:
        result.update({
            'hard_v2_triggers': int(hard_counts.sum()),
            'examples_with_hard_v2_trigger': int((hard_counts > 0).sum()),
            'mean_hard_v2_triggers': float(hard_counts.mean()),
        })
    else:
        result.update({
            'hard_v2_triggers': None,
            'examples_with_hard_v2_trigger': None,
            'mean_hard_v2_triggers': None,
        })
    return result


trigger_rows = [
    trigger_summary(shared_triggers, 'shared_selector'),
    trigger_summary(dual_epoch1_triggers, 'dual_epoch1'),
    trigger_summary(dual_epoch2_triggers, 'dual_epoch2'),
    trigger_summary(shared_hard_triggers, 'shared_selector_hard_v2'),
    trigger_summary(dual_epoch1_hard_triggers, 'dual_epoch1_hard_v2'),
    trigger_summary(dual_epoch2_hard_triggers, 'dual_epoch2_hard_v2'),
]

trigger_frame = pd.DataFrame(trigger_rows)
display(trigger_frame)

trigger_frame.to_csv(
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_triggers.csv'),
    index=False,
)
atomic_json_dump(
    trigger_rows,
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_triggers.json'),
)


## 14. Qualitative changed-example audit

In [ ]:
def changed_example_rows(candidate_name, baseline_name, limit=50):
    candidate = ARMS[candidate_name]
    baseline = ARMS[baseline_name]
    rows = []

    for index, item in enumerate(ITEMS):
        if candidate[index] == baseline[index]:
            continue
        candidate_score = grounding_score(candidate[index], item['ex']['triples'])
        baseline_score = grounding_score(baseline[index], item['ex']['triples'])
        rows.append({
            'uid': item['uid'],
            'raw_test_index': item['idx'],
            'unseen': item['unseen'],
            'triples': json.dumps([_tp(t) for t in item['ex']['triples']], ensure_ascii=False),
            'baseline': baseline[index],
            'candidate': candidate[index],
            'recall_delta': candidate_score['recall'] - baseline_score['recall'],
            'halluc_delta': candidate_score['halluc'] - baseline_score['halluc'],
            'baseline_corruptions': json.dumps(baseline_score['corruptions'], ensure_ascii=False),
            'candidate_corruptions': json.dumps(candidate_score['corruptions'], ensure_ascii=False),
        })

    rows.sort(key=lambda row: (-abs(row['recall_delta']), -abs(row['halluc_delta'])))
    return rows[:limit], rows


audit_specs = {
    'dual_e2_vs_fusion': ('dual_epoch2', 'fusion_only'),
    'dual_e2_vs_shared': ('dual_epoch2', 'shared_selector'),
    'dual_e2_vs_dual_e1': ('dual_epoch2', 'dual_epoch1'),
    'dual_e2_hard_vs_fusion_hard': ('dual_epoch2_hard_v2', 'fusion_hard_v2'),
    'dual_e2_hard_vs_shared_hard': ('dual_epoch2_hard_v2', 'shared_selector_hard_v2'),
    'dual_e2_hard_vs_dual_e1_hard': ('dual_epoch2_hard_v2', 'dual_epoch1_hard_v2'),
}

for audit_name, (candidate_name, baseline_name) in audit_specs.items():
    preview, all_rows = changed_example_rows(candidate_name, baseline_name)
    pd.DataFrame(all_rows).to_csv(
        os.path.join(OUTPUT_DIR, f'changed_examples_{audit_name}.csv'),
        index=False,
    )
    print('\n', audit_name, '| changed outputs:', len(all_rows))
    display(pd.DataFrame(preview))


## 15. Decision summary

In [ ]:
def overall_delta(candidate_name, baseline_name):
    candidate = TEST_SUMMARY['arms'][candidate_name]['overall']
    baseline = TEST_SUMMARY['arms'][baseline_name]['overall']
    return {
        'BLEU': candidate['bleu'] - baseline['bleu'],
        'recall_pp': 100 * (candidate['recall'] - baseline['recall']),
        'halluc_pp': 100 * (candidate['halluc'] - baseline['halluc']),
        'corruption_rows': candidate['corr_rows'] - baseline['corr_rows'],
    }


decision = {
    'dual_e2_vs_fusion': overall_delta('dual_epoch2', 'fusion_only'),
    'dual_e2_vs_shared': overall_delta('dual_epoch2', 'shared_selector'),
    'dual_e2_vs_dual_e1': overall_delta('dual_epoch2', 'dual_epoch1'),
    'dual_e2_hard_vs_fusion_hard': overall_delta(
        'dual_epoch2_hard_v2', 'fusion_hard_v2'),
    'dual_e2_hard_vs_shared_hard': overall_delta(
        'dual_epoch2_hard_v2', 'shared_selector_hard_v2'),
    'dual_e2_hard_vs_dual_e1_hard': overall_delta(
        'dual_epoch2_hard_v2', 'dual_epoch1_hard_v2'),
}

def positive_recall_ci(comparison):
    return PAIRED_RESULTS[comparison]['overall']['recall']['ci95'][0] > 0

decision['evidence_flags'] = {
    'dual_e2_recall_beats_fusion': positive_recall_ci('dual_e2_vs_fusion'),
    'dual_e2_recall_beats_shared': positive_recall_ci('dual_e2_vs_shared'),
    'dual_e2_recall_beats_dual_e1': positive_recall_ci('dual_e2_vs_dual_e1'),
    'dual_e2_hard_recall_beats_fusion_hard': positive_recall_ci(
        'dual_e2_hard_vs_fusion_hard'),
    'dual_e2_hard_recall_beats_shared_hard': positive_recall_ci(
        'dual_e2_hard_vs_shared_hard'),
    'dual_e2_hard_recall_beats_dual_e1_hard': positive_recall_ci(
        'dual_e2_hard_vs_dual_e1_hard'),
}

atomic_json_dump(
    decision,
    os.path.join(OUTPUT_DIR, 'dual_head_epoch2_repaired_test_decision.json'),
)
print(json.dumps(decision, indent=2))

print('\nInterpretation guide:')
print('- Epoch 2 > shared: the calibrated residual dual head demonstrates descriptive test value.')
print('- Epoch 2 ≈ shared: the dual head survives a fair comparison but has not earned its complexity.')
print('- Epoch 2 < shared: development calibration did not fully resolve the test deficit.')
print('- Treat this as post-development descriptive evidence, not untouched confirmation.')


## 16. Package outputs

In [ ]:
run_manifest = {
    'notebook': '20_dual_head_epoch2_repaired_test_comparison_colab.ipynb',
    'scientific_status': (
        'one frozen post-development descriptive evaluation of the '
        'development-selected epoch-2 checkpoint; not pristine confirmatory'
    ),
    'dual_epoch2_checkpoint': {
        'path': DUAL_EPOCH2_PATH,
        'epoch': DUAL_EPOCH,
        'sha256': DUAL_EPOCH2_SHA,
        'selection_source': SELECTED_OPERATING_POINT_PATH,
    },
    'current_selector': {
        'path': CURRENT_SELECTOR_CKPT,
        'sha256': CURRENT_SELECTOR_SHA,
    },
    'fusion_checkpoint': CKPT_FUSION,
    'repaired_test_order_sha256': REPAIRED_ORDER_SHA,
    'frozen_config': {
        'selector_tau': SELECTOR_TAU,
        'compatibility_margin': COMPATIBILITY_MARGIN,
        'hard_v2_min_depth': LOCK_MIN_DEPTH,
        'hard_v2_escape_margin': ESCAPE_MARGIN,
        'bootstrap_samples': BOOTSTRAP_SAMPLES,
    },
    'reused_prediction_files': {
        'fusion_only': fusion_path,
        'fusion_hard_v2': fusion_hard_path,
        'shared_selector': SHARED_PRED_PATH,
        'shared_selector_hard_v2': SHARED_HARD_PRED_PATH,
        'dual_epoch1': PREVIOUS_EPOCH1_PRED_PATH,
        'dual_epoch1_hard_v2': PREVIOUS_EPOCH1_HARD_PRED_PATH,
    },
    'new_generation_arms': [
        'dual_epoch2',
        'dual_epoch2_hard_v2',
    ],
}
atomic_json_dump(
    run_manifest,
    os.path.join(OUTPUT_DIR, 'run_manifest.json'),
)

archive_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Output archive:', archive_path)
print('\nOutput files:')
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(' ', filename)
